In [ ]:
# -----------------------------
# Import statements
# -----------------------------

# --- computer imports ---
from pathlib import Path
import os
import re
import pandas as pd

# --- data analysis imports ---
import matplotlib.pyplot as plt

import numpy as np
from numpy import genfromtxt

from scipy.special import factorial
from scipy.stats import poisson, binomtest, norm
from scipy.stats import chi2 as chi2_dist
from scipy.optimize import curve_fit
from scipy.special import erf

In [ ]:
# -----------------------------
# Inputs: results from calibration curve fit
# -----------------------------

CAL_DIR = Path("plots/calibration")

cal_re = re.compile(
    r"run(\d+)_slope=([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)_b=([+-]?\d+(?:\.\d+)?(?:e[+-]?\d+)?)",
    re.IGNORECASE
)

cal_m1 = cal_b1 = cal_m2 = cal_b2 = None

for file in CAL_DIR.glob("calcurve_run*.png"):
    m = cal_re.search(file.name)
    if not m:
        continue

    run = int(m.group(1))
    slope = float(m.group(2))
    intercept = float(m.group(3))

    if run == 1:
        cal_m1 = slope
        cal_b1 = intercept
    elif run == 2:
        cal_m2 = slope
        cal_b2 = intercept

# (debug)
print("Calibration constants loaded:")
print(f"Run 1: m = {cal_m1}, b = {cal_b1}")
print(f"Run 2: m = {cal_m2}, b = {cal_b2}")

if None in (cal_m1, cal_b1, cal_m2, cal_b2):
    raise RuntimeError("One or more calibration constants were not found.")

In [ ]:
# -----------------------------
# Inputs:
# -----------------------------

In [ ]:
# -----------------------------
# HELPER FUNCTIONS
# -----------------------------

def exp_only(t, A, tau):
    return A * np.exp(-t / tau)

def exp_plus_bkg(t, A, tau, B):
    return A * np.exp(-t / tau) + B

def chisq(y, y_model, sigma):
    return np.sum(((y - y_model) / sigma)**2)

def chisq_and_red(y, y_model, sigma, n_params):
    chi2_val = np.sum(((y - y_model) / sigma)**2)
    dof = len(y) - n_params
    return chi2_val, dof, chi2_val / dof

def nested_chi2_test(chi2_null, chi2_alt, dof_null, dof_alt):
    """
    Null: simpler model (fewer params), Alt: more params
    For nested models, Delta chi^2 ~ chi^2_{Delta dof}.
    Returns (delta, delta_dof, p_value).
    """
    delta = chi2_null - chi2_alt
    delta_dof = dof_null - dof_alt
    # Guard: if numeric noise makes delta negative, clamp to 0
    delta = max(delta, 0.0)
    p = 1.0 - chi2_dist.cdf(delta, df=delta_dof)
    return delta, delta_dof, p

def plot_fit_with_residuals(t, y, sigma, model_func, popt, title, fname=None):
    """
    two-panel plot:
      top: data with error bars + model curve
      bottom: normalized residuals (y - model)/sigma
    """
    y_model = model_func(t, *popt)
    resid = (y - y_model) / sigma

    fig = plt.figure(figsize=(14, 7.5))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.08)

    ax1 = fig.add_subplot(gs[0])
    ax1.errorbar(t, y, yerr=sigma, fmt='o', markersize=3, label="Data")
    t_smooth = np.linspace(t.min(), t.max(), 600)
    ax1.plot(t_smooth, model_func(t_smooth, *popt), '-', label="Fit")
    ax1.set_ylabel("Counts per bin")
    ax1.set_title(title)
    ax1.legend()

    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.axhline(0, linewidth=1)
    ax2.plot(t, resid, 'o', markersize=3)
    ax2.set_xlabel("Decay time (μs)")
    ax2.set_ylabel("Res/σ")

    if fname is not None:
        fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.show()


def analyze_muon_run(
    run_label: str,
    data_path: str | Path,
    cal_m: float,
    cal_b: float,
    out_dir: str | Path,
    adc_cut=(125, 4075),
    nbins=200
):
    """
    - Reads muon data file with columns: index, ADC
    - Applies acceptance cut in ADC (same as your older workflow)
    - Converts ADC->time using calibration (m,b): t_us = ((ADC - b)/m)/1000
    - Makes plots (raw fine, accepted rebinned, log-linear)
    - Fits pure exponential A exp(-t/tau) on accepted rebinned histogram (no background term)
    - Saves all plots into out_dir
    """
    data_path = Path(data_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    muon = np.loadtxt(data_path)
    adc = muon[:, 1].astype(float)

    # ----------------------------
    # Raw fine histogram in ADC channels (x-axis displayed in μs)
    # ----------------------------
    adc_min, adc_max = 0, 4096
    bins_fine = np.arange(adc_min - 0.5, adc_max + 1.5, 1)
    counts_fine, edges_fine = np.histogram(adc, bins=bins_fine)
    centers_fine_adc = 0.5 * (edges_fine[1:] + edges_fine[:-1])

    centers_fine_us = ((centers_fine_adc - cal_b) / cal_m) / 1000.0  # μs

    plt.figure(figsize=(18, 7))
    plt.step(centers_fine_us, counts_fine, where="mid")
    plt.xlabel("Decay time (μs)")
    plt.ylabel("Counts per (ADC channel)")
    plt.title(f"{run_label}: raw distribution (fine bins)")
    plt.savefig(out_dir / "raw_muon_fine.png", dpi=300, bbox_inches="tight")
    plt.show()

    # ----------------------------
    # Acceptance cut in ADC (same as old code)
    # ----------------------------
    lo, hi = adc_cut
    mask = (adc >= lo) & (adc <= hi)
    adc_acc = adc[mask]

    # ----------------------------
    # Rebin accepted data into nbins in ADC-space; display x-axis in μs
    # ----------------------------
    edges = np.linspace(lo, hi, nbins + 1)
    counts, edges = np.histogram(adc_acc, bins=edges)
    centers_adc = 0.5 * (edges[1:] + edges[:-1])
    centers_us = ((centers_adc - cal_b) / cal_m) / 1000.0

    sigma = np.sqrt(np.maximum(counts, 1.0))

    plt.figure(figsize=(14, 7.5))
    plt.errorbar(centers_us, counts, yerr=sigma, fmt='o', markersize=3)
    plt.xlabel("Decay time (μs)")
    plt.ylabel(f"Counts per bin ({nbins} bins)")
    plt.title(f"{run_label}: accepted muon decay distribution ({nbins} bins)")
    plt.savefig(out_dir / f"masked_muon_{nbins}bins.png", dpi=300, bbox_inches="tight")
    plt.show()

    # ----------------------------
    # Log-linear plot (no background subtraction)
    # ----------------------------
    pos = counts > 0
    plt.figure(figsize=(14, 7.5))
    plt.errorbar(
        centers_us[pos],
        np.log(counts[pos]),
        yerr=sigma[pos] / counts[pos],
        fmt='o',
        markersize=3
    )
    plt.xlabel("Decay time (μs)")
    plt.ylabel("log(Counts per bin)")
    plt.title(f"{run_label}: log-linear plot (accepted, {nbins} bins)")
    plt.savefig(out_dir / "masked_muon_log_lin.png", dpi=300, bbox_inches="tight")
    plt.show()

    # ----------------------------
    # Fit region: use bins with counts>0 (and inside acceptance by construction)
    # Model: A exp(-t/tau) (NO +B)
    # ----------------------------
    t_fit = centers_us[pos]
    y_fit = counts[pos].astype(float)
    s_fit = sigma[pos]

    # Reasonable initial guesses
    p0 = [y_fit.max(), 2.0]  # A, tau (μs)

    popt, pcov = curve_fit(
        exp_only,
        t_fit,
        y_fit,
        sigma=s_fit,
        absolute_sigma=True,
        p0=p0
    )

    A_hat, tau_hat = popt
    A_err, tau_err = np.sqrt(np.diag(pcov))

    y_model = exp_only(t_fit, *popt)
    chi2_val = chisq(y_fit, y_model, s_fit)
    dof = len(t_fit) - 2
    red_chi2 = chi2_val / dof

    print(f"\n[{run_label}] pure exponential fit: y = A exp(-t/tau)")
    print(f"A   = {A_hat:.6g} ± {A_err:.2g}")
    print(f"tau = {tau_hat:.6g} ± {tau_err:.2g}  μs   (95%: ± {1.96*tau_err:.2g} μs)")
    print(f"chi^2 = {chi2_val:.3f}, dof = {dof}, reduced chi^2 = {red_chi2:.3f}")

    # Overlay + residuals plot
    plot_fit_with_residuals(
        t_fit, y_fit, s_fit,
        model_func=exp_only,
        popt=popt,
        title=f"{run_label}: Fit A exp(-t/τ) (no background term)",
        fname=out_dir / "fit_overlay_residuals.png"
    )

    # Return numbers so you can store/use them downstream
    return {
        "A": A_hat, "A_err": A_err,
        "tau_us": tau_hat, "tau_err_us": tau_err,
        "chi2": chi2_val, "dof": dof, "red_chi2": red_chi2
    }

In [ ]:
# -----------------------------
# RESULTS
# -----------------------------

res1 = analyze_muon_run(
    run_label="Run 1",
    data_path="data/RUN1 JAN 30-FEB 3 2026.txt",
    cal_m=cal_m1,
    cal_b=cal_b1,
    out_dir="plots/run 1"
)

res2 = analyze_muon_run(
    run_label="Run 2",
    data_path="data/RUN2 FEB 3-FEB 6 2026.txt",
    cal_m=cal_m2,
    cal_b=cal_b2,
    out_dir="plots/run 2"
)